In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
 
spark = (
    SparkSession.builder
    .appName("gold-showcase")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
    .config("spark.sql.defaultCatalog", "lakehouse")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")

Spark 4.1.0


In [2]:
drivers = spark.read.table("lakehouse.cdc.silver_drivers").count()
zones = spark.read.table("lakehouse.taxi.gold_demand_patterns").select("pickup_zone").distinct().count()
avg_demand = spark.sql("SELECT AVG(avg_trip_count) as avg FROM lakehouse.taxi.gold_demand_patterns").collect()[0]["avg"]

print(f"Total drivers:       {drivers}")
print(f"Total zones:         {zones}")
print(f"Drivers per zone:    {drivers / zones:.4f}")
print(f"Avg demand per zone: {avg_demand:.4f}")

Total drivers:       126
Total zones:         128
Drivers per zone:    0.9844
Avg demand per zone: 43.3146


In [3]:
print(" Demand patterns overview ")
spark.sql("""
    SELECT
        pickup_zone,
        hour_of_day,
        ROUND(avg_trip_count, 2)    AS avg_trips,
        ROUND(stddev_trip_count, 2) AS std_dev,
        demand_classification
    FROM lakehouse.taxi.gold_demand_patterns
    ORDER BY avg_trips DESC
    LIMIT 10
""").show(truncate=False)

 Demand patterns overview 
+---------------------+-----------+---------+-------+---------------------+
|pickup_zone          |hour_of_day|avg_trips|std_dev|demand_classification|
+---------------------+-----------+---------+-------+---------------------+
|East Village         |1          |330.0    |0.0    |high demand          |
|Lincoln Square East  |0          |306.0    |0.0    |high demand          |
|East Village         |0          |267.0    |0.0    |high demand          |
|Midtown Center       |0          |228.0    |0.0    |high demand          |
|Gramercy             |1          |225.0    |0.0    |high demand          |
|Upper East Side South|0          |225.0    |0.0    |high demand          |
|West Village         |0          |214.0    |0.0    |high demand          |
|Upper West Side South|0          |214.0    |0.0    |high demand          |
|Midtown Center       |1          |207.0    |0.0    |high demand          |
|JFK Airport          |0          |207.0    |0.0    |high dem

In [4]:
print("Distribution of demand classifications")
spark.sql("""
    SELECT
        demand_classification,
        COUNT(*) AS zone_hour_combinations
    FROM lakehouse.taxi.gold_demand_patterns
    GROUP BY demand_classification
    ORDER BY zone_hour_combinations DESC
""").show()


Distribution of demand classifications
+---------------------+----------------------+
|demand_classification|zone_hour_combinations|
+---------------------+----------------------+
|               normal|                   256|
|          high demand|                    46|
+---------------------+----------------------+



In [5]:
print("Which 3 zones have the most predictable demand?")
spark.sql("""
    SELECT
        pickup_zone,
        ROUND(AVG(stddev_trip_count), 4) AS avg_std_dev,
        ROUND(AVG(avg_trip_count), 2)    AS avg_trips_per_hour,
        COUNT(*)                         AS hours_observed
    FROM lakehouse.taxi.gold_demand_patterns
    GROUP BY pickup_zone
    HAVING COUNT(*) >= 3
    ORDER BY avg_std_dev ASC
    LIMIT 3
""").show(truncate=False)

Which 3 zones have the most predictable demand?
+---------------------+-----------+------------------+--------------+
|pickup_zone          |avg_std_dev|avg_trips_per_hour|hours_observed|
+---------------------+-----------+------------------+--------------+
|Upper East Side South|0.0        |155.67            |3             |
|Yorkville West       |0.0        |151.33            |3             |
|TriBeCa/Civic Center |0.0        |74.0              |3             |
+---------------------+-----------+------------------+--------------+



In [6]:
print("At what hour does demand peak city-wide?")
spark.sql("""
    SELECT
        hour_of_day,
        ROUND(SUM(avg_trip_count), 0)  AS total_avg_trips_citywide,
        COUNT(DISTINCT pickup_zone)    AS zones_active
    FROM lakehouse.taxi.gold_demand_patterns
    GROUP BY hour_of_day
    ORDER BY total_avg_trips_citywide DESC
""").show(24, truncate=False)

At what hour does demand peak city-wide?
+-----------+------------------------+------------+
|hour_of_day|total_avg_trips_citywide|zones_active|
+-----------+------------------------+------------+
|0          |5545.0                  |94          |
|1          |5393.0                  |113         |
|2          |2125.0                  |80          |
|23         |14.0                    |11          |
|20         |2.0                     |2           |
|3          |1.0                     |1           |
|21         |1.0                     |1           |
+-----------+------------------------+------------+



In [7]:
print("Supply-demand gap — underserved zones")
spark.sql("""
    SELECT
        pickup_zone,
        hour_of_day,
        ROUND(avg_trip_count, 2)     AS avg_demand,
        ROUND(drivers_available, 2)  AS drivers_available,
        ROUND(demand_supply_gap, 2)  AS gap,
        is_underserved
    FROM lakehouse.taxi.gold_supply_demand_gap
    WHERE is_underserved = TRUE
    ORDER BY demand_supply_gap DESC
    LIMIT 15
""").show(truncate=False)

Supply-demand gap — underserved zones
+-----------------------+-----------+----------+-----------------+------+--------------+
|pickup_zone            |hour_of_day|avg_demand|drivers_available|gap   |is_underserved|
+-----------------------+-----------+----------+-----------------+------+--------------+
|East Village           |1          |330.0     |7.71             |322.29|true          |
|Lincoln Square East    |0          |306.0     |6.95             |299.05|true          |
|East Village           |0          |267.0     |6.07             |260.93|true          |
|Midtown Center         |0          |228.0     |5.18             |222.82|true          |
|Upper East Side South  |0          |225.0     |5.11             |219.89|true          |
|Gramercy               |1          |225.0     |5.26             |219.74|true          |
|West Village           |0          |214.0     |4.86             |209.14|true          |
|Upper West Side South  |0          |214.0     |4.86             |209.14

In [8]:
total        = spark.read.table("lakehouse.taxi.gold_demand_patterns").count()
underserved  = spark.sql("SELECT COUNT(*) as c FROM lakehouse.taxi.gold_supply_demand_gap WHERE is_underserved = TRUE").collect()[0]["c"]
total_gaps   = spark.read.table("lakehouse.taxi.gold_supply_demand_gap").count()
print(f"Total zone/hour combinations analysed : {total}")
print(f"Underserved zone/hour combinations    : {underserved} / {total_gaps}")
print(f"Underserved percentage                : {round(underserved/total_gaps*100, 1)}%")

Total zone/hour combinations analysed : 302
Underserved zone/hour combinations    : 287 / 302
Underserved percentage                : 95.0%
